# 01 · Lectura e integración de Work-Olist

De archivos tabulares a datos comprensibles: rutas, tipos, unidades de observación, filtros, agregaciones y uniones relacionales seguras.

**Modalidad:** práctica guiada, código ejecutable, interpretación y ejercicios. **Datos:** nueve CSV aportados por el usuario en `data/raw/`, con el esquema relacional de Olist. Se calculan resultados sobre estos archivos, sin simular registros ni cambiar las fuentes.

## Objetivos y prerrequisitos
Al finalizar podrás: (i) distinguir archivo, tabla, fila, columna y variable; (ii) importar CSV y diagnosticar tipos; (iii) reconocer llaves y cardinalidades; (iv) filtrar, ordenar, agrupar y unir sin duplicar pedidos; (v) exportar un resultado con trazabilidad. Se explican `Series`, `DataFrame`, `list`, `dict`, `loc`, `groupby` y `merge` sin suponer experiencia previa.

**Procedencia y acceso.** Los nueve CSV se entregaron en este proyecto en `data/raw/`. Su esquema es el publicado para el [conjunto Brazilian E-Commerce de Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce). El análisis utiliza los archivos suministrados, no presupone una descarga ni verifica su identidad byte a byte con Kaggle. El proyecto conserva los nombres de los CSV recibidos y unifica las operaciones en `src/olist_tools.py`.

**Pregunta guía:** ¿cuántos pedidos, compradores y líneas de producto hay, y qué pasa si confundimos sus unidades de análisis?

In [ ]:
from pathlib import Path
import sys
from IPython.display import display
import numpy as np
import pandas as pd
# Detectar carpeta del proyecto desde raíz/ o notebooks/ (sin rutas absolutas).
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "src" / "olist_tools.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Ejecute el notebook dentro de la carpeta Work_Olist_Notebooks_2026 (README.md).")
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.olist_tools import (read_tables, build_order_level, build_item_level,
                            table_inventory, missing_profile, safe_minmax, safe_zscore)
TABLAS, ORIGEN = read_tables(ROOT)
print("ORIGEN:", ORIGEN)
print("Directorio de trabajo:", ROOT)

### Diccionario relacional y unidades de observación

| Tabla | Unidad de observación | Llave o criterio de identidad | Precaución principal |
|---|---|---|---|
| `orders` | un pedido | `order_id` | el estado condiciona la disponibilidad de fechas |
| `customers` | registro de cliente vinculado al pedido | `customer_id` | no equivale a `customer_unique_id` |
| `items` | línea de pedido | (`order_id`, `order_item_id`) | varios ítems por pedido |
| `payments` | registro de pago | (`order_id`, `payment_sequential`) | varios pagos por pedido |
| `reviews` | reseña asociada a pedido | (`review_id`, `order_id`) | puede haber varias reseñas por pedido |
| `products` | producto | `product_id` | categorías y atributos físicos pueden faltar |
| `sellers` | vendedor | `seller_id` | estado es etiqueta geográfica |
| `translation` | categoría traducida | `product_category_name` | no asumir cobertura de todas las categorías |
| `geolocation` | coordenada registrada para prefijo postal | no hay llave única de fila | un prefijo postal puede tener muchos puntos |

**Advertencia sobre unidades:** sumar pagos sobre una tabla expandida por ítems duplica importes. Agregar primero las tablas de detalle por `order_id` preserva una fila por pedido.


## 1. Reconocer los archivos antes de leer los datos
Una ruta relativa permite trabajar en distintas computadoras. En Python, `Path` representa rutas; `pandas.read_csv` lee tablas y conserva los identificadores como texto para no confundirlos con cantidades. Un CSV no guarda información de tipos tan rica como una base relacional: debemos diagnosticar después de importar.

In [ ]:
# Datos reales: conteos y llaves comprobables en los archivos aportados.
resumen_fuente = pd.DataFrame([
    {"tabla": nombre, "registros": len(df), "columnas": df.shape[1]}
    for nombre, df in TABLAS.items()
]).sort_values("tabla")
display(resumen_fuente)
assert TABLAS["orders"].order_id.is_unique
assert TABLAS["customers"].customer_id.is_unique
print("Compradores distintos (ID anonimizado):", TABLAS["customers"].customer_unique_id.nunique())

In [ ]:
print("Archivos esperados (solo nombres):")
for nombre, tabla in TABLAS.items():
    print(f"  {nombre:12s}: {tabla.shape[0]:>7,} filas × {tabla.shape[1]} columnas")
print("\nInventario y memoria aproximada:")
display(table_inventory(TABLAS))

### 1.1. Novena tabla: geolocalización sin sobrecargar memoria

Las ocho primeras tablas pueden integrarse por claves comerciales. La novena, `geolocation`, tiene **1 000 163 registros** y múltiples coordenadas para un mismo prefijo postal. En esta primera lectura mostramos únicamente cinco filas y contamos líneas en el CSV, **sin unir cada punto a los pedidos**. En el notebook 3 aprenderemos a resumir coordenadas antes de visualizarlas.

Un archivo extenso no exige cargarlo entero para conocer su estructura. Si más adelante se requieren cálculos geográficos, puede usarse `read_csv(usecols=...)`, lectura por bloques (`chunksize`) o agregación por prefijo.

In [ ]:
# Inspección eficiente de geolocation: solo encabezado, cinco filas y conteo físico.
archivo_geo = ROOT / "data" / "raw" / "olist_geolocation_dataset.csv"
print("Geolocalización: filas =", sum(1 for _ in archivo_geo.open(encoding="utf-8")) - 1)
display(pd.read_csv(archivo_geo, nrows=5))
print("No se realiza ninguna unión geográfica en esta etapa.")

## 2. Vistas previas, estructura y significado
`head()` devuelve las primeras filas; `info()` resume tipos y nulos; `describe()` describe magnitudes y categorías. **No confundir número almacenado con magnitud:** un prefijo postal no debe promediarse. No mostrar aquí los textos libres de reseñas.

In [ ]:
pedidos = TABLAS["orders"].copy()
clientes = TABLAS["customers"].copy()
items = TABLAS["items"].copy()
pagos = TABLAS["payments"].copy()
display(pedidos.head(4))
print("\nTipos y completitud de pedidos:")
pedidos.info()
display(missing_profile(pedidos).head(9))
print("\nResumen de precios de líneas de pedido (BRL, sin flete):")
display(items["price"].describe(percentiles=[.1,.25,.5,.75,.9,.99]).to_frame())

### Tipos de datos y estructuras computacionales
Una **lista** preserva orden, un **diccionario** asocia claves con valores, una **tupla** es una secuencia inmutable y un `numpy.ndarray` permite cálculo vectorizado. `Series` y `DataFrame` agregan etiquetas y valores faltantes. Esta cultura computacional sirve para elegir estructuras sencillas antes de crear clases o patrones complejos.

In [ ]:
columnas = ["order_id", "order_status"]       # lista
config = {"tabla":"orders", "llave":"order_id"} # diccionario
etapas = ("leer", "auditar", "unir")             # tupla
precios = items["price"].dropna().to_numpy()     # ndarray
print(type(columnas).__name__, type(config).__name__, type(etapas).__name__, type(precios).__name__)
print("Promedio vectorizado (ilustración, no limpieza):", np.mean(precios).round(2))

## 3. Identificadores, unidades y cardinalidad
`order_id` identifica un pedido, `customer_id` identifica el registro de cliente asociado al pedido y `customer_unique_id` aproxima un comprador recurrente. En `items`, `payments` y `reviews` es legítimo encontrar varias filas por pedido. Una unión `one_to_many` multiplica filas; si luego sumamos `payment_value` desde esa tabla expandida, podemos duplicar ingresos **por error**.

In [ ]:
chequeos = pd.DataFrame([
  {"tabla":"orders","llave":"order_id","filas":len(pedidos),"llaves_unicas":pedidos.order_id.nunique()},
  {"tabla":"items","llave":"order_id + order_item_id","filas":len(items),
   "llaves_unicas":items[["order_id","order_item_id"]].drop_duplicates().shape[0]},
  {"tabla":"payments","llave":"order_id + payment_sequential","filas":len(pagos),
   "llaves_unicas":pagos[["order_id","payment_sequential"]].drop_duplicates().shape[0]},
  {"tabla":"customers","llave":"customer_id","filas":len(clientes),"llaves_unicas":clientes.customer_id.nunique()},
])
display(chequeos)
print("Pedidos / registros de cliente / compradores únicos:",
      pedidos.order_id.nunique(),clientes.customer_id.nunique(),clientes.customer_unique_id.nunique())
assert pedidos.order_id.is_unique

## 4. Filtrar, ordenar, seleccionar y crear variables
`loc[filas, columnas]` deja explícito el criterio de selección; `.assign()` genera columnas sin sobrescribir innecesariamente los datos originales. Las condiciones se combinan con `&` (y), `|` (o) y paréntesis. Evita `for` fila por fila cuando una expresión vectorizada resuelve el problema.

In [ ]:
# Un filtro por condición y columnas, sin modificar "pedidos".
sp_entregados = pedidos.loc[
    pedidos["order_status"].eq("delivered") &
    pedidos["customer_id"].isin(clientes.loc[clientes["customer_state"].eq("SP"), "customer_id"]),
    ["order_id","customer_id","order_purchase_timestamp","order_status"]
].sort_values("order_purchase_timestamp")
print("Entregados en SP:",len(sp_entregados))
display(sp_entregados.head(5))
lineas = items.assign(valor_con_flete=lambda d: d["price"] + d["freight_value"])
display(lineas[["order_id","order_item_id","price","freight_value","valor_con_flete"]].head())

## 5. Groupby, resumen y tablas de doble entrada
`groupby` separa, aplica y combina. Para comparar estados es necesario recordar que los conteos son **pedidos**, no personas; los estados no necesariamente tienen la misma población. `crosstab` es útil con variables nominales u ordinales.

In [ ]:
por_estado = (pedidos.merge(clientes[["customer_id","customer_state"]],on="customer_id",validate="many_to_one")
              .groupby("customer_state").agg(pedidos=("order_id","nunique"))
              .sort_values("pedidos",ascending=False))
display(por_estado.head(10))
ct = pd.crosstab(pedidos["order_status"], columns="pedidos", normalize="columns")
display(ct.rename(columns={"pedidos":"proporcion"}).round(3))

## 6. Uniones con `validate` e `indicator`
**Caso seguro:** unir pedidos con clientes mediante `many_to_one`. **Caso inseguro:** unir pedidos con ítems y pagos a la vez sin agregación previa. `indicator=True` ayuda a identificar ausencias en las llaves y `validate=` detecta relaciones no previstas. Para construir una tabla analítica de pedidos agregaremos ítems, pagos y reseñas **antes** de unir.

In [ ]:
prueba_merge = pedidos[["order_id","customer_id"]].merge(
    clientes[["customer_id","customer_state"]], on="customer_id", how="left",
    validate="many_to_one", indicator=True)
display(prueba_merge["_merge"].value_counts(dropna=False).to_frame("filas"))
analitica = build_order_level(TABLAS)
print("Filas en pedidos originales:",len(pedidos),"| filas en tabla analítica:",len(analitica))
assert len(analitica)==len(pedidos)
assert analitica.order_id.is_unique
print("Cada fila sigue representando un pedido.")
display(analitica[["order_id","n_items","n_pagos","subtotal_productos","total_flete","total_pagado"]].head())

### Por qué las sumas pueden no coincidir exactamente
`sum(order_items.price)` es el subtotal de productos y **no** incluye flete; `sum(payment_value)` puede incorporar flete, descuentos, reembolsos u otras diferencias operativas. La tabla original no debe forzarse a igualdad por conveniencia. Los pedidos cancelados y estados incompletos requieren interpretación separada.

In [ ]:
comparacion = analitica[["subtotal_productos","total_flete","total_pagado"]].sum(min_count=1)
display(comparacion.rename("total_BRL").round(2).to_frame())
# La tabla de líneas mantiene una unidad distinta.
items_categorias = build_item_level(TABLAS)
print("Filas en ítems (antes/después de unir catálogos):",len(items),len(items_categorias))
assert len(items_categorias)==len(items)
display(items_categorias[["order_id","product_id","categoria_mostrada","price"]].head())

## 7. Fechas y variables derivadas
Convertir fechas con `pd.to_datetime(errors='coerce')` permite construir duración, mes y retraso. **No interpretes NA en fecha de entrega como cero días**: un pedido no entregado no tiene duración observada. La interpretación temporal depende también de convenciones horarias y eventos de negocio.

In [ ]:
columnas = ["order_id","order_status","mes_compra","dias_hasta_entrega","dias_retraso","entrega_tardia"]
display(analitica[columnas].head(8))
print("Tasa tardía entre pedidos con duración observada:",
      f"{analitica['entrega_tardia'].dropna().mean():.1%}")

## 8. Una función reutilizable y exportación trazable
Una función expresa contrato, entradas y salidas. No se necesita programación orientada a objetos para una transformación tabular sencilla. Guardaremos el resultado derivado aparte de los CSV originales, sin sobrescribir las fuentes. El `README` explica cómo reproducirlo.

In [ ]:
def resumen_por_estado(df: pd.DataFrame) -> pd.DataFrame:
    """Cuenta pedidos por estado sin confundirlos con clientes únicos."""
    return (df.groupby("customer_state",dropna=False)
              .agg(pedidos=("order_id","nunique"),
                   compradores=("customer_unique_id","nunique"),
                   mediana_valor=("subtotal_productos","median"))
              .sort_values("pedidos",ascending=False))
display(resumen_por_estado(analitica).head(8).round(2))
from json import dumps
outdir = ROOT / "data" / "processed"; outdir.mkdir(parents=True,exist_ok=True)
analitica.to_csv(outdir / "olist_pedidos_analitica.csv",index=False)
(outdir / "origen_tabla_analitica.txt").write_text(
    f"Origen: {ORIGEN}\nTransformación: src.olist_tools.build_order_level\nUnidad: pedido\n",encoding="utf8")
print("Guardado en data/processed/ (las fuentes originales no se modificaron).")

### Interpretación con los datos suministrados

En los archivos recibidos aparecen **99 441 pedidos**, **112 650 líneas de pedido** y **103 886 registros de pago**. Esta diferencia no es un error: las tablas tienen **distintas unidades de observación**. La tabla de clientes registra 99 441 identificadores `customer_id`, pero 96 096 identificadores `customer_unique_id` distintos; por ello no debemos llamar «personas» al conteo de registros de clientes.

**Pregunta de reflexión:** ¿por qué una fusión entre pedidos, ítems y pagos, sin agregaciones previas, no permite sumar de manera fiable los importes de compra?


## Interpretación y límites
Una unión correcta no garantiza conclusiones correctas: revisa la población analizada, la cobertura temporal, el contexto y el hecho de que datos de clientes, pedidos, pagos e ítems tienen granularidades distintas. Los resultados numéricos corresponden a los CSV suministrados; las conclusiones son descriptivas, no representativas de todo el comercio electrónico brasileño.

## Ejercicios (para resolver fuera de clase)
- **Esencial.** Filtra los pedidos `delivered` y calcula el conteo por mes. Contrasta dos métodos (`value_counts` y `groupby`).
- **Interpretación.** Explica por qué no se debe sumar `payment_value` después de unir directamente una tabla de pagos a una de ítems.
- **Extensión computacional.** Construye una función que reciba un código de estado y devuelva pedidos, compradores distintos y valor mediano. Agrega una aserción para validar que el filtro corresponde a un estado conocido.

**Referencias:** notas de las presentaciones anteriores (naturaleza, tipos, calidad, implementación práctica); programa docente aprobado; documentación de [pandas](https://pandas.pydata.org/docs/user_guide/index.html) y [Olist/Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce).

In [ ]:
# Espacio de trabajo (escriba su solución sin modificar las celdas previas):
# 1) Su agrupación por mes aquí.
# 2) Su argumento sobre el doble conteo en Markdown.
# 3) def resumen_estado(df, estado): ...